In [38]:
import pandas as pd
import numpy as np
from IPython.display import Math

In [2]:
training_set = pd.read_csv('/home/lukas/Desktop/digit-recognizer/train.csv')
test = pd.read_csv('/home/lukas/Desktop/digit-recognizer/test.csv')

In [3]:
train_set = training_set.drop(['label'], axis=1)
labels = training_set['label'].to_numpy()

In [4]:
labels = training_set['label']

In [6]:
def ReLU(x):
    x = np.maximum(0, x)
    return x

In [24]:
def Softmax(x):
    x_exp = np.exp(x)
    output = x_exp / np.sum(x_exp)

    return output

In [26]:
def layer(weights, bias, x_in, func):
    
    z = x_in @ weights + bias
    activation = func(z)

    return z, activation

In [27]:
def init_parameters(layer_neurons):
    num_layers = len(layer_neurons)
    list_weights = []
    list_bias = []
    
    for i in range(1, num_layers):
        n_in = layer_neurons[i - 1]
        n_out = layer_neurons[i]

        weight = np.random.randn(n_in, n_out) * np.sqrt(2 / n_in)
        list_weights.append(weight)
        list_bias.append(np.zeros((1, n_out)))

    return list_weights, list_bias

In [28]:
layer_neurons = [784, 10, 10, 10]
w, b = init_parameters(layer_neurons)

In [52]:
def forward_prop(x_in, w, b):
    num_layers = len(w)
    activation_dict = {}
    z_dict = {}
    
    for i in range(num_layers - 1):
        z, a = layer(w[i], b[i], x_in, ReLU)
        x_in = a
        z_dict[i + 1] = z
        activation_dict[i + 1] = a
        
    z, a = layer(w[-1], b[-1], z, Softmax)
    z_dict[num_layers] = z
    activation_dict[num_layers] = a
    
    return activation_dict, z_dict

In [54]:
a_dict, z_dict = forward_prop(test, w, b)

In [56]:
def compute_cost(x_in, label):
    
    idx = np.argmax(label)            
    x = x_in[0]
    
    return -np.log(x[idx] + 1e-12)   

In [57]:
def ReLU_derivative(z):
    return (z > 0).astype(float)

In [218]:

def backpropagation(w, b, activation_dict, z_dict, y, x_in):
    num_operations = len(w)
    gradients_w = [None] * num_operations
    gradients_b = [None] * num_operations

    ###  Pochodna dC/dw
    A_out = activation_dict[num_operations] ### Funkcja aktywacji którą wyrzuca output layer
    A_prev = activation_dict[num_operations - 1] ### Funkcja aktywacji która trafia do output layer

    # dC/da(L)
    gamma = (A_out - y) * ReLU_derivative(z_dict[num_operations])

    gradients_w[-1] = A_prev.T @ gamma
    gradients_b[-1] = np.sum(gamma, axis = 0, keepdims=True)


    for L in reversed(range(1, num_operations)):

        if L == 1:
            A_prev = x_in
        else:
            A_prev = activation_dict[L]

        # gamma z następnej warstwy * waga następnej warstwy * pochodna funkcji aktywacji aktualnej warstwy
        gamma = (gamma @ w[L].T) * ReLU_derivative(z_dict[L])
        gradients_w[L-1] = A_prev.T @ gamma
        gradients_b[L-1] = np.sum(gamma, axis = 0, keepdims=True)


    return gradients_w, gradients_b
    

In [221]:
x_in = np.random.randn(1, 784)
labels = np.zeros((1, 10))

In [224]:
gradients_w, gradients_b = backpropagation(w, b, a_dict, z_dict, labels, x_in)

In [225]:
w[0]

array([[-0.04527962, -0.04384491,  0.00904924, ...,  0.11005314,
         0.05731444,  0.0923686 ],
       [-0.04825702, -0.00602671, -0.03086175, ..., -0.02959506,
         0.02204173,  0.03854736],
       [ 0.04403924, -0.00731795,  0.05120619, ..., -0.109158  ,
         0.00331055, -0.03331183],
       ...,
       [-0.03064601, -0.01997366, -0.027927  , ...,  0.00438431,
         0.01749382,  0.01463141],
       [ 0.00518066, -0.07683183,  0.1056642 , ..., -0.03335258,
         0.04439722,  0.06313126],
       [-0.03687332,  0.01164635,  0.04872262, ..., -0.08934045,
        -0.11074143,  0.02876817]], shape=(784, 10))